set up
1. 安装Mortal-main

    1.1. 进入Mortal-main目录

    1.2. 执行命令：`cargo build`

    1.3. 进入libriichi目录，执行命令：`
            cargo build --release --features pymod`
        此时venv\lib\site-packages\libriichi.pyd已经生成,如果没生成需要复制过去。
2. pip install numpy,torch,matplotlib,tensorboard


work-flow-training
1. 使用get_uuids.py抓取牌谱,得到一个uuid列表文件
2. 使用download_browser.mjs抓取牌谱，得到一个tenhou.net/6格式牌谱文件夹
3. 使用tenhou2mjai.py将tenhou.net/6格式牌谱转换为mjai格式牌谱
4. 使用export_decision_points.py从mjai格式牌谱提取相应玩家的决策点数据集（因为需要使用libriichi中的函数，需要把export_decision_points.py放在libriichi\mortal目录下执行）
5. 使用train.py训练模型



Notes

1. tenhou.net/6格式牌谱是按照牌局的自然发展顺序记录的，能清楚知道摸牌、牌河、鸣牌等信息。而mjai是按照决策来记录的，更加适合训练。
2. 经过试验，常用账号一天能抓取150局左右的牌谱。新号只能抓取10局左右的牌谱。
3. 在300局牌谱的训练集上训练，模型的准确率大约在0.7。提高到500，准确率在0.73左右。我猜测，训练集的数量越多，模型的准确率也不会有显著提升。首先随着抓取牌谱的范围变大，果圣的打牌风格也会有变化。其次很多局牌的决策点实际上有多个平行选择的，所以采用top k的方式训练模型应该会更加贴合实际。
4. 仅供娱乐，千万不要用这个模型去和ai打来观测果圣的水平啊。


Top K训练模型
根据Note，我使用了Top k的方式对模型重新训练。具体代码参见train_top_k.py。出发点是因为在训练集里，很多局牌的决策点实际上有多个平行选择的，比如手牌是（1s，3s，4s，1p，3p，4p)，假设牌河也是对称的，这种情况下1s和1p应该是处于相同的地位。但是模型只选一个，而在实战中，这几乎是随机选择的。所以只看概率最高的决策大概率是存在瓶颈的。这一点从实验中也可以看出来。

为此，我在训练模型时，提取前k个概率最高的决策点，再进行归一化。并将以0.15权重加到原logits向量上。
即：$$0.85 * [v_1,...,v_k,v_{k+1},...,v_n] + 0.15 * [v_1^{\prime},\cdots,v_k^{\prime},0,...]$$
这样最高decision的概率会被降低，其他次高决策的概率会被提高。这样训练出来的模型会更加贴近实际。
在loss_curve_topk.png中可以看到，top2的概率已经到了0.9以上，top3的概率达到0.95以上。

### 雀魂抓取
抓取功能每个账号每天？只能抓取100局牌谱左右，所以请使用多个账号抓取牌谱。而且一旦达到上限，在玩游戏的时候是没办法看牌谱回放了。
